# Phishing URL Detection

This notebook trains a machine learning model to classify URLs as **phishing** or **legitimate**.

**Model**: Random Forest Classifier  
**Features**: 53 URL-based handcrafted features  
**Dataset**: `dataset_phishing.csv` (11,430 URLs, balanced classes)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

print("Libraries loaded successfully!")


## 1. Load and Inspect Data


In [ ]:
df = pd.read_csv('dataset_phishing.csv')
print(f"Dataset shape: {df.shape}")
print(f"Columns: {len(df.columns)}")
print(df['status'].value_counts())


## 2. Basic EDA


In [ ]:
# Check class balance
sns.countplot(x='status', data=df)
plt.title('Distribution of URL Status')
plt.show()

# Check for missing values
print(df.isnull().sum().sum(), 'missing values')
print(df.duplicated().sum(), 'duplicate rows')


## 3. Preprocess Data

Map labels to numeric values and split features from target. We'll train on URL-only features (no page-content data).


In [ ]:
df['status'] = df['status'].map({'legitimate': 0, 'phishing': 1})

# URL-only features used for prediction
url_features = [
    'length_url', 'length_hostname', 'ip', 'nb_dots', 'nb_hyphens', 'nb_at',
    'nb_qm', 'nb_and', 'nb_eq', 'nb_underscore', 'nb_tilde', 'nb_percent',
    'nb_slash', 'nb_colon', 'nb_comma', 'nb_semicolumn', 'nb_dollar',
    'nb_space', 'nb_www', 'nb_com', 'nb_dslash', 'http_in_path',
    'https_token', 'ratio_digits_url', 'ratio_digits_host', 'punycode',
    'port', 'tld_in_path', 'tld_in_subdomain', 'abnormal_subdomain',
    'nb_subdomains', 'prefix_suffix', 'random_domain', 'shortening_service',
    'path_extension', 'nb_redirection', 'nb_external_redirection',
    'length_words_raw', 'char_repeat', 'shortest_words_raw', 'shortest_word_host',
    'shortest_word_path', 'longest_words_raw', 'longest_word_host',
    'longest_word_path', 'avg_words_raw', 'avg_word_host', 'avg_word_path',
    'phish_hints', 'domain_in_brand', 'brand_in_subdomain', 'brand_in_path',
    'suspecious_tld'
]

missing = [col for col in url_features if col not in df.columns]
print('Missing columns:', missing)

X = df[url_features]
y = df['status']
print('Feature matrix shape:', X.shape)
print('Target shape:', y.shape)


## 4. Train/Test Split


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train:', X_train.shape)
print('Test:', X_test.shape)


## 5. Train Random Forest

We use Random Forest because it handles non-linear relationships well, is robust to outliers, and provides feature importance. A GridSearchCV step tunes `n_estimators` and `max_depth`.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None]
}

rf = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(
    rf, param_grid, cv=5, scoring='accuracy', n_jobs=-1
)
grid_search.fit(X_train, y_train)

print('Best parameters:', grid_search.best_params_)
print('Best CV accuracy:', round(grid_search.best_score_, 4))

best_rf = grid_search.best_estimator_


## 6. Evaluate Model


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = best_rf.predict(X_test)
print('Test Accuracy:', round(accuracy_score(y_test, y_pred), 4))
print()
print(classification_report(y_test, y_pred))
print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))


## 7. Feature Importance

Inspect which URL features drive the model's decisions.


In [ ]:
importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': best_rf.feature_importances_
}).sort_values('importance', ascending=False)

print(importance_df.head(15).to_string(index=False))


## 8. Save Model

Export the final model and feature schema for the web API.


In [ ]:
import joblib

joblib.dump(best_rf, 'Phishing_url_model.pkl')
joblib.dump(url_features, 'phishing_url_features.pkl')

print('Model and features saved successfully!')


## 9. Test Inference

Run a quick sanity check on a sample URL.


In [ ]:
from urllib.parse import urlparse
import re

def extract_features(url):
    features = {}
    features['length_url'] = len(url)
    features['nb_dots'] = url.count('.')
    features['nb_hyphens'] = url.count('-')
    features['nb_at'] = url.count('@')
    features['nb_qm'] = url.count('?')
    features['nb_and'] = url.count('&')
    features['nb_eq'] = url.count('=')
    features['nb_underscore'] = url.count('_')
    features['nb_tilde'] = url.count('~')
    features['nb_percent'] = url.count('%')
    features['nb_slash'] = url.count('/')
    features['nb_colon'] = url.count(':')
    features['nb_comma'] = url.count(',')
    features['nb_semicolumn'] = url.count(';')
    features['nb_dollar'] = url.count('$')
    features['nb_space'] = url.count(' ')
    features['nb_com'] = url.lower().count('.com')

    parsed_url = urlparse(url)
    hostname = parsed_url.hostname or ''
    path = parsed_url.path
    hostname_lower = hostname.lower()
    path_lower = path.lower()
    url_lower = url.lower()

    features['length_hostname'] = len(hostname)
    features['nb_www'] = hostname_lower.count('www')
    features['nb_subdomains'] = max(hostname.count('.') - 1, 0)

    features['nb_dslash'] = max(url.count('//') - 1, 0)
    features['http_in_path'] = int('http' in path_lower)
    features['https_token'] = int('https' in url_lower)
    features['ratio_digits_url'] = sum(c.isdigit() for c in url) / len(url) if len(url) > 0 else 0
    features['ratio_digits_host'] = sum(c.isdigit() for c in hostname) / len(hostname) if len(hostname) > 0 else 0

    url_words = url.replace('/', ' ').replace('.', ' ').replace('-', ' ').split()
    host_words = hostname.replace('.', ' ').replace('-', ' ').split()
    path_words = path.replace('/', ' ').replace('-', ' ').split()

    features['length_words_raw'] = len(url_words)
    features['shortest_words_raw'] = min([len(w) for w in url_words], default=0)
    features['shortest_word_host'] = min([len(w) for w in host_words], default=0)
    features['shortest_word_path'] = min([len(w) for w in path_words], default=0)
    features['longest_words_raw'] = max([len(w) for w in url_words], default=0)
    features['longest_word_host'] = max([len(w) for w in host_words], default=0)
    features['longest_word_path'] = max([len(w) for w in path_words], default=0)
    features['avg_words_raw'] = sum(len(w) for w in url_words) / len(url_words) if url_words else 0
    features['avg_word_host'] = sum(len(w) for w in host_words) / len(host_words) if host_words else 0
    features['avg_word_path'] = sum(len(w) for w in path_words) / len(path_words) if path_words else 0

    features['char_repeat'] = max([url.count(c) for c in set(url)], default=0)
    features['prefix_suffix'] = int('-' in hostname)

    shortening_services = ['bit.ly', 'tinyurl.com', 'goo.gl', 't.co', 'ow.ly', 'is.gd', 'buff.ly', 'adf.ly', 'bit.do']
    features['shortening_service'] = int(any(s in hostname_lower for s in shortening_services))

    suspicious_tlds = ['.tk', '.ml', '.ga', '.cf', '.gq', '.top', '.xyz', '.club']
    features['suspecious_tld'] = int(any(hostname_lower.endswith(tld) for tld in suspicious_tlds))

    features['ip'] = int(re.match(r'^\d{1,3}(\.\d{1,3}){3}$', hostname) is not None)
    try:
        features['port'] = int(parsed_url.port is not None)
    except ValueError:
        features['port'] = 0

    hostname_parts = hostname.split('.')
    tld = '.' + hostname_parts[-1] if len(hostname_parts) >= 2 else ''
    subdomain = '.'.join(hostname_parts[:-2])

    features['tld_in_path'] = int(tld != '' and tld in path_lower)
    features['tld_in_subdomain'] = int(tld != '' and tld in subdomain.lower())
    features['abnormal_subdomain'] = int(len(hostname_parts) > 4)
    features['random_domain'] = int(bool(re.search(r'[0-9]{4,}', hostname)))

    features['punycode'] = int('xn--' in hostname_lower)
    features['path_extension'] = int(bool(re.search(r'\.[a-zA-Z0-9]{1,5}$', path)))
    features['nb_redirection'] = url.count('redirect')
    features['nb_external_redirection'] = url.count('http://') + url.count('https://') - 1

    brands = ['google', 'facebook', 'paypal', 'amazon', 'microsoft', 'apple', 'netflix', 'instagram', 'linkedin', 'twitter', 'bank']
    features['domain_in_brand'] = int(any(b in hostname_lower for b in brands))
    features['brand_in_subdomain'] = int(any(b in subdomain.lower() for b in brands))
    features['brand_in_path'] = int(any(b in path_lower for b in brands))

    suspicious_words = ['login', 'signin', 'verify', 'verification', 'account', 'update', 'secure', 'security', 'confirm', 'password', 'bank', 'paypal']
    features['phish_hints'] = sum(word in url_lower for word in suspicious_words)

    return features

def predict_url(url):
    features = extract_features(url)
    features_df = pd.DataFrame([features])[url_features]
    pred = best_rf.predict(features_df)[0]
    prob = best_rf.predict_proba(features_df)[0]
    label = 'PHISHING' if pred == 1 else 'LEGITIMATE'
    confidence = round(prob[pred] * 100, 2)
    print(f'URL: {url}')
    print(f'Prediction: {label}')
    print(f'Confidence: {confidence}%')

predict_url('https://www.example.com/login')
predict_url('http://paypal-secure.verify-account.com/signin')
